# 26. Демонстрация чат-бота SRA
Используя архитектуру материнской платы из Notebook 25, мы создаем пользовательский интерфейс чата с помощью ipywidgets.
На основе ввода пользователя маршрутизатор в реальном времени выбирает правильный синапс и выдает ответ.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import ipywidgets as widgets
from IPython.display import display, clear_output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LLM_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Using device: {device}")


In [ ]:
class LLMSynapse(nn.Module):
    """General-purpose LLM synapse (Neural)"""
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model)
        )
        # Load the Qwen Instruct model
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(LLM_MODEL_ID).to(device)
        self.model.eval()
    
    def forward(self, x, text_input=None):
        if text_input is not None:
            # Feed the user input through Qwen's chat format and generate a response
            system_message = {
                "role": "system",
                "content": "You are having a normal one-on-one conversation. Reply only to the user's latest message in one short natural sentence. Do not introduce yourself. Do not explain what you are. Do not write code, tutorials, numbered lists, or examples unless explicitly asked. Match the user's language.",
            }
            if isinstance(text_input, list):
                messages = [system_message] + text_input[-8:]
            else:
                messages = [system_message, {"role": "user", "content": text_input}]
            prompt = self.tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=False,
            )
            encoded = self.tokenizer(prompt, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = self.model.generate(
                    input_ids=encoded["input_ids"],
                    attention_mask=encoded["attention_mask"],
                    max_new_tokens=40,
                    do_sample=False,
                    repetition_penalty=1.1,
                    eos_token_id=self.tokenizer.eos_token_id,
                    pad_token_id=self.tokenizer.eos_token_id,
                )
            generated = outputs[0, encoded["input_ids"].shape[-1]:]
            response = self.tokenizer.decode(generated, skip_special_tokens=True).strip()
            for marker in ["```", "\n- ", "\n1.", "\n2.", "\n3."]:
                response = response.split(marker, 1)[0].strip()
            response = response.split("\n\n", 1)[0].strip()
            return response
        return self.net(x)

In [ ]:
class VectorDBSynapse(nn.Module):
    """Fact-retrieval synapse -- a true vector DB with cosine-similarity search.

    Each fact (key, value) is embedded into a fixed semantic vector. Queries
    are embedded the same way and matched by cosine similarity. The
    `embedder` is supplied externally (e.g. the LLM's input-embedding layer)
    so this class stays agnostic about which encoder is used.
    """
    def __init__(self, d_model, embedder=None):
        super().__init__()
        self.is_neural = False
        self.db = {}            # key -> value (text)
        self.d_model = d_model
        self.embedder = embedder
        self._embeds = {}       # key -> normalized 1D torch.Tensor

    def set_embedder(self, embedder):
        """Attach an embedder (callable: str -> 1D normalized tensor) and re-embed all facts."""
        self.embedder = embedder
        self._embeds = {k: embedder(f"{k} {v}") for k, v in self.db.items()}

    def add_fact(self, key, value):
        self.db[key] = value
        if self.embedder is not None:
            self._embeds[key] = self.embedder(f"{key} {value}")

    def search(self, query, top_k=5):
        """Cosine-similarity search. Returns [(key, value, sim), ...] sorted desc."""
        if self.embedder is None or not self._embeds:
            return []
        q = self.embedder(query)
        scored = []
        for k, kv in self._embeds.items():
            sim = float((q * kv).sum())
            scored.append((sim, k))
        scored.sort(reverse=True)
        return [(k, self.db[k], s) for s, k in scored[:top_k]]

    def forward(self, x, text_input=None):
        if text_input is not None:
            # 1) Exact substring match (cheap, deterministic)
            for k, v in self.db.items():
                if k.lower() in text_input.lower():
                    return f"DB_RESULT: {v}"
            # 2) Semantic similarity fallback
            hits = self.search(text_input, top_k=1)
            if hits and hits[0][2] >= 0.4:
                k, v, sim = hits[0]
                return f"DB_RESULT: {v} (similar to '{k}', sim={sim:.2f})"
            return "DB_MISS: not found in the knowledge base"
        return x

In [ ]:
class CalculatorSynapse(nn.Module):
    """Rule-based calculator synapse"""
    def __init__(self):
        super().__init__()
        self.is_neural = False
    def forward(self, x, text_input=None):
        if text_input is not None:
            try:
                ans = eval(text_input)
                return f"CALC_RESULT: {ans}"
            except:
                return "CALC_ERROR: not a valid arithmetic expression"
        return x

In [ ]:
class LastTokenRouter(nn.Module):
    def __init__(self, d_model, num_synapses):
        super().__init__()
        self.classifier = nn.Linear(d_model, num_synapses, bias=False)
        
    def forward(self, x):
        last_token_embeds = x[:, -1, :]
        logits = self.classifier(last_token_embeds)
        probs = F.softmax(logits, dim=-1)
        return probs, logits

class SRAMotherboard(nn.Module):
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.synapses = nn.ModuleDict()
        self.synapse_names = []
        self.router = None 
        
    def add_synapse(self, name, synapse_module):
        self.synapses[name] = synapse_module
        self.synapse_names.append(name)
        old_router = self.router
        self.router = LastTokenRouter(self.d_model, len(self.synapse_names)).to(device)
        if old_router is not None:
            with torch.no_grad():
                self.router.classifier.weight[:len(self.synapse_names)-1, :] = old_router.classifier.weight.data

    def _is_failed_output(self, output):
        return isinstance(output, str) and output.startswith(("CALC_ERROR", "DB_MISS"))

    def _extract_math_expression(self, text):
        import re
        pattern = r'[\d\s\+\-\*\/\(\)\.]+'
        matches = re.findall(pattern, text)
        for match in matches:
            clean = match.strip()
            if any(op in clean for op in ['+', '-', '*', '/']) and len(clean) >= 3:
                return clean
        return None

    def _llm_generate(self, llm_syn, messages, max_new_tokens=150):
        """Bypass LLMSynapse's generation cap and run inference directly with the requested token budget."""
        prompt = llm_syn.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False
        )
        encoded = llm_syn.tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = llm_syn.model.generate(
                input_ids=encoded["input_ids"],
                attention_mask=encoded["attention_mask"],
                max_new_tokens=max_new_tokens,
                do_sample=False,
                eos_token_id=llm_syn.tokenizer.eos_token_id,
                pad_token_id=llm_syn.tokenizer.eos_token_id,
            )
        generated = outputs[0, encoded["input_ids"].shape[-1]:]
        return llm_syn.tokenizer.decode(generated, skip_special_tokens=True).strip()

    # ----- VectorDB-driven helpers (no hardcoded prices / items / discounts) -----
    @staticmethod
    def _parse_price(fact_value):
        """Extract the first yen amount from a VectorDB fact value."""
        import re
        if not isinstance(fact_value, str):
            return None
        # NOTE: matches the Japanese yen character; the demo dataset is in Japanese.
        m = re.search(r'(\d+)\s*円', fact_value)
        return int(m.group(1)) if m else None

    @staticmethod
    def _parse_counter(fact_value):
        """Extract the counter unit (Japanese counters: 個, 本, 房, 杯, 枚, 台, ...) from a VectorDB fact value."""
        import re
        if not isinstance(fact_value, str):
            return None
        # NOTE: Japanese counters are part of the experimental scenario.
        m = re.search(r'\d+\s*([個本房杯枚台缶袋瓶箱])', fact_value)
        return m.group(1) if m else None

    @staticmethod
    def _parse_discount_rule(fact_value):
        """Detect a discount rule in a VectorDB fact and return (kind, value).

        Supported patterns:
          - "0.9倍" or "1.2倍"                   -> ('multiplier', float)
          - "10%オフ" / "10%off" / "10%割引"   -> ('multiplier', 1 - pct/100)
          - "50円引き"                             -> ('subtract',   int)
        Returns None if no discount rule is found.
        """
        import re
        if not isinstance(fact_value, str):
            return None
        # NOTE: Japanese discount-rule patterns are part of the experimental scenario.
        m = re.search(r'(\d+(?:\.\d+)?)\s*倍', fact_value)
        if m:
            return ('multiplier', float(m.group(1)))
        m = re.search(r'(\d+)\s*円\s*引き', fact_value)
        if m:
            return ('subtract', int(m.group(1)))
        m = re.search(r'(\d+(?:\.\d+)?)\s*%\s*(?:オフ|off|割引)', fact_value, re.IGNORECASE)
        if m:
            return ('multiplier', 1.0 - float(m.group(1)) / 100.0)
        return None

    def _collect_vdb_catalog(self):
        """Walk the VectorDB once and split facts into priced items vs. discount rules."""
        items = {}      # name -> {'price': int, 'counter': str|None, 'raw': str}
        discounts = {}  # name -> {'kind': str, 'value': float|int, 'raw': str}
        if "VectorDB" not in self.synapses:
            return items, discounts
        vdb = self.synapses["VectorDB"]
        for key, value in vdb.db.items():
            rule = self._parse_discount_rule(value)
            if rule is not None:
                discounts[key] = {'kind': rule[0], 'value': rule[1], 'raw': value}
                continue
            price = self._parse_price(value)
            if price is not None:
                items[key] = {'price': price, 'counter': self._parse_counter(value), 'raw': value}
        return items, discounts

    def _looks_like_shopping_request(self, text):
        """Heuristic gate: only fire cooperative routing when the user is shopping.

        NOTE: the trigger keywords are Japanese because the demo scenario is a
        Japanese-language fruit-shop dialogue.
        """
        if not text:
            return False
        return any(k in text for k in ["買", "個", "本", "合計", "いくら", "円", "キャンペーン", "クーポン"])

    def _canonicalize_name(self, raw_name, candidates, kind="商品", min_sim=0.15):
        """Resolve a free-form (possibly descriptive) name to one of `candidates`
        using **only** VectorDB's similarity search.

        1) exact match  2) substring match  3) `VectorDB.search` similarity
        Falls back to None if no candidate is reachable. No LLM reasoning,
        no hardcoded synonym tables -- the VDB is the entire knowledge source.
        """
        if not isinstance(raw_name, str) or not raw_name.strip():
            return None
        if raw_name in candidates:
            return raw_name
        for c in candidates:
            if c and (c in raw_name or raw_name in c):
                return c
        if not candidates:
            return None
        if "VectorDB" not in self.synapses:
            return None
        vdb = self.synapses["VectorDB"]
        if not hasattr(vdb, "search"):
            return None
        try:
            hits = vdb.search(raw_name, top_k=5)
        except Exception as e:
            print(f"[SRA Motherboard] VDB.search failed for '{raw_name}': {e}")
            return None
        for hit_key, hit_value, sim in hits:
            if sim < min_sim:
                break
            # (a) direct hit: the VDB key itself is a canonical candidate
            if hit_key in candidates:
                print(f"[SRA Motherboard] VDB-sim '{raw_name}' -> '{hit_key}' (sim={sim:.3f}, direct)")
                return hit_key
            # (b) indirect hit: the matched fact references a canonical name in its text
            combined = f"{hit_key} {hit_value}"
            best_ref = None
            for c in candidates:
                if c and c in combined:
                    if best_ref is None or len(c) > len(best_ref):
                        best_ref = c
            if best_ref is not None:
                print(f"[SRA Motherboard] VDB-sim '{raw_name}' -> '{best_ref}' (via '{hit_key}', sim={sim:.3f})")
                return best_ref
        return None

    def route_and_forward(self, input_ids, text_inputs=None, llm_messages=None):
        x = self.embedding(input_ids)
        probs, _ = self.router(x)
        
        batch_size = x.size(0)
        final_outputs = []
        routed_synapses = []
        
        for i in range(batch_size):
            text_input = text_inputs[i] if text_inputs else None
            
            # Multi-stage intelligent cooperative routing (VectorDB-driven, no hardcoding)
            if (
                self._looks_like_shopping_request(text_input)
                and "GeneralLLM" in self.synapses
                and "Calculator" in self.synapses
            ):
                available_items, available_discounts = self._collect_vdb_catalog()
                if available_items:
                    print("[SRA Motherboard] Multi-stage cooperative routing triggered!")

                    # Phase 1: Build an extraction prompt from the live VectorDB catalog.
                    # NOTE: the prompt below is in Japanese to match the experimental scenario
                    # (Japanese-language fruit-shop dialogue with Qwen).
                    items_section = "\n".join(
                        f"- {name}: {meta['raw']}" for name, meta in available_items.items()
                    )
                    discounts_section = "\n".join(
                        f"- {name}: {meta['raw']}" for name, meta in available_discounts.items()
                    ) or "（なし）"

                    item_names = list(available_items.keys())
                    discount_names = list(available_discounts.keys())
                    if len(item_names) >= 2:
                        ex_items = f'{{"{item_names[0]}": 1, "{item_names[1]}": 2}}'
                    else:
                        ex_items = f'{{"{item_names[0]}": 1}}'
                    ex_discounts = f'["{discount_names[0]}"]' if discount_names else "[]"
                    example_block = (
                        f'例)\n'
                        f'JSON: {{"items": {ex_items}, "discounts": {ex_discounts}}}'
                    )

                    extraction_prompt = (
                        "タスク: ユーザーの注文を、下の商品リストと割引リストにある正式名称へマッピングし、"
                        "JSONのみを出力してください。\n"
                        "色・形・俗称など説明的な言い方は、商品リスト内の正式名称に正規化すること。"
                        "リストに存在しない商品名・割引名は出力禁止です。\n\n"
                        f"商品リスト:\n{items_section}\n\n"
                        f"割引リスト:\n{discounts_section}\n\n"
                        f"{example_block}\n\n"
                        f"ユーザー: {text_input}\n"
                        "JSON:"
                    )

                    llm_syn = self.synapses["GeneralLLM"]
                    raw_json = self._llm_generate(
                        llm_syn,
                        [
                            {"role": "system", "content": "You are a precise data extractor. Output ONLY a clean JSON block. No explanation."},
                            {"role": "user", "content": extraction_prompt},
                        ],
                        max_new_tokens=200,
                    )
                    print(f"[SRA Motherboard] LLM Raw JSON Output: {raw_json}")

                    try:
                        import json as _json
                        import re as _re
                        json_match = _re.search(r'\{.*\}', raw_json, _re.DOTALL)
                        if not json_match:
                            raise ValueError("No JSON block found")
                        parsed = _json.loads(json_match.group(0))
                        items_ordered = parsed.get("items", {}) or {}
                        discounts_applied = parsed.get("discounts", []) or []

                        # Phase 2: Build the formula straight from VectorDB prices
                        terms = []
                        item_lines = []
                        item_candidates = list(available_items.keys())
                        for raw_item_name, qty in items_ordered.items():
                            try:
                                qty_int = int(qty)
                            except (TypeError, ValueError):
                                continue
                            if qty_int <= 0:
                                continue
                            canonical = self._canonicalize_name(
                                raw_item_name, item_candidates, kind="商品"
                            )
                            if canonical is None:
                                print(f"[SRA Motherboard] Skip unknown item: '{raw_item_name}'")
                                continue
                            if canonical != raw_item_name:
                                print(f"[SRA Motherboard] Canonicalize item: '{raw_item_name}' -> '{canonical}'")
                            price = available_items[canonical]['price']
                            counter = available_items[canonical]['counter'] or "個"
                            terms.append(f"{price} * {qty_int}")
                            item_lines.append(f"{canonical} {qty_int}{counter} ({price * qty_int}円)")

                        if not terms:
                            raise ValueError("No valid items extracted from user request")

                        formula = "(" + " + ".join(terms) + ")"
                        discount_lines = []
                        discount_candidates = list(available_discounts.keys())
                        for raw_disc_name in discounts_applied:
                            canonical_disc = self._canonicalize_name(
                                raw_disc_name, discount_candidates, kind="割引"
                            )
                            if canonical_disc is None:
                                print(f"[SRA Motherboard] Skip unknown discount: '{raw_disc_name}'")
                                continue
                            if canonical_disc != raw_disc_name:
                                print(f"[SRA Motherboard] Canonicalize discount: '{raw_disc_name}' -> '{canonical_disc}'")
                            rule = available_discounts[canonical_disc]
                            if rule['kind'] == 'multiplier':
                                formula = f"({formula}) * {rule['value']}"
                            elif rule['kind'] == 'subtract':
                                formula = f"({formula}) - {rule['value']}"
                            discount_lines.append(f"{canonical_disc} ({rule['raw']})")

                        print(f"[SRA Motherboard] Dynamically built formula: {formula}")

                        # Phase 3: Calculator computes the exact total
                        calc_out = self.synapses["Calculator"](x[i:i+1], text_input=formula)
                        if self._is_failed_output(calc_out):
                            raise ValueError(f"Calculation failed: {calc_out}")
                        calc_result = calc_out.replace("CALC_RESULT: ", "").strip()
                        # Drop trailing ".0" from integer-valued floats so totals read naturally
                        if calc_result.endswith(".0"):
                            calc_result = calc_result[:-2]

                        # Build the breakdown text in Python from VectorDB data (no hardcoding)
                        items_str = "、".join(item_lines)
                        discounts_str = "、".join(discount_lines) if discount_lines else "なし"

                        # Phase 4: Final LLM response -- shopkeeper role-play + few-shot.
                        # NOTE: the user-facing prompt below is Japanese because the demo is a
                        # Japanese-language fruit-shop dialogue.
                        final_messages = [
                            {
                                "role": "system",
                                "content": (
                                    "You are a friendly fruit shop clerk. Respond in natural Japanese. "
                                    "Use ONLY the exact total amount provided. Do not recalculate."
                                ),
                            },
                            {
                                "role": "user",
                                "content": (
                                    f"以下の情報をもとに、お客様に合計金額をお伝えする、自然でおもてなしあふれる一文を日本語で生成してください。\n\n"
                                    f"【注文内容】{items_str}\n"
                                    f"【適用割引】{discounts_str}\n"
                                    f"【合計金額】{calc_result}円（この金額は正確です。再計算しないでください。）\n\n"
                                    f"出力例: 「{items_str}で、{discounts_str}を適用し、合計 {calc_result}円 でございます。ありがとうございます！」\n"
                                    f"あなたの応答:"
                                ),
                            },
                        ]
                        out = self._llm_generate(llm_syn, final_messages, max_new_tokens=150)
                        final_outputs.append(f"[Cooperative(LLM->VectorDB->Calculator->LLM)] {out}")
                        routed_synapses.append("Cooperative(LLM+DB+Calc)")
                        continue

                    except Exception as e:
                        print(f"[SRA Motherboard] Error: {e}. Falling back to normal routing.")

            # Standard dynamic routing fallback
            math_expr = self._extract_math_expression(text_input) if text_input else None
            calc_result = None
            
            if math_expr and "Calculator" in self.synapses:
                calc_out = self.synapses["Calculator"](x[i:i+1], text_input=math_expr)
                if not self._is_failed_output(calc_out):
                    calc_result = calc_out.replace("CALC_RESULT: ", "").strip()
            
            if calc_result and "GeneralLLM" in self.synapses:
                # NOTE: hint is in Japanese to match the demo scenario.
                cooperative_hint = (
                    f"(参考情報: システムの計算モジュールが算出した結果、数式 '{math_expr}' の正解は '{calc_result}' です。"
                    f"この値を用いて、ユーザーの指示に自然な日本語で答えてください。自分で再計算しないでください。)"
                )
                synapse_input = text_input
                if llm_messages is not None and len(llm_messages) > i:
                    adjusted_messages = list(llm_messages[i])
                    if adjusted_messages and adjusted_messages[-1]["role"] == "user":
                        adjusted_messages[-1] = {
                            "role": "user",
                            "content": f"{adjusted_messages[-1]['content']}\n{cooperative_hint}"
                        }
                    synapse_input = adjusted_messages
                else:
                    synapse_input = f"{text_input}\n{cooperative_hint}"
                
                out = self.synapses["GeneralLLM"](x[i:i+1], text_input=synapse_input)
                final_outputs.append(f"[Cooperative(Calculator->GeneralLLM)] {out}")
                routed_synapses.append("Cooperative(Calc+LLM)")
                continue
            
            sample_probs = probs[i]
            sorted_indices = torch.argsort(sample_probs, descending=True)
            
            for rank, syn_idx in enumerate(sorted_indices):
                selected_synapse_name = self.synapse_names[syn_idx.item()]
                synapse = self.synapses[selected_synapse_name]
                synapse_input = text_input
                if selected_synapse_name == "GeneralLLM" and llm_messages is not None:
                    synapse_input = llm_messages[i]
                out = synapse(x[i:i+1], text_input=synapse_input)
                if self._is_failed_output(out):
                    continue
                prefix = f"[{selected_synapse_name}]"
                if rank > 0:
                    primary_name = self.synapse_names[sorted_indices[0].item()]
                    prefix = f"[{selected_synapse_name} fallback from {primary_name}]"
                final_outputs.append(f"{prefix} {out}")
                routed_synapses.append(selected_synapse_name)
                break
            else:
                primary_name = self.synapse_names[sorted_indices[0].item()]
                # NOTE: user-facing error stays in Japanese to match the demo scenario.
                final_outputs.append(f"[{primary_name}] 全シナプスで処理に失敗しました")
                routed_synapses.append(primary_name)
        
        return final_outputs, routed_synapses[-1] if len(routed_synapses) == 1 else routed_synapses


In [ ]:
vocab_size = 1000
d_model = 128

model = SRAMotherboard(d_model=d_model, vocab_size=vocab_size).to(device)

# Register synapses
model.add_synapse("GeneralLLM", LLMSynapse(d_model).to(device))
model.add_synapse("Calculator", CalculatorSynapse())

# Build a lightweight embedder from Qwen's input-embedding layer (no forward pass).
# This is used by VectorDB as its semantic encoder -- Qwen acts here only as a
# vocabulary-level vector source, NOT as a reasoner.
_llm_syn = model.synapses["GeneralLLM"]
_tokenizer = _llm_syn.tokenizer
_input_embed_layer = _llm_syn.model.get_input_embeddings()

def _qwen_input_embedder(text):
    enc = _tokenizer(text, return_tensors="pt", truncation=True, max_length=64).to(device)
    with torch.no_grad():
        token_vecs = _input_embed_layer(enc["input_ids"])
    mask = enc["attention_mask"].unsqueeze(-1).float()
    pooled = (token_vecs * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return F.normalize(pooled, dim=-1).squeeze(0)

vdb = VectorDBSynapse(d_model, embedder=_qwen_input_embedder)
vdb.add_fact("Japan", "Tokyo")
vdb.add_fact("CEO", "John Doe")
# NOTE: the following demo facts stay in Japanese because the chatbot's
# experimental scenario is a Japanese fruit-shop dialogue. The price parser
# (_parse_price), counter parser (_parse_counter) and discount parser
# (_parse_discount_rule) match on Japanese characters (円, 個, 倍, etc.),
# so translating these facts would break the multi-stage cooperative routing demo.
vdb.add_fact("SynapticRouter", "動的に最適なモジュールへルーティングする次世代AIアーキテクチャです。")

# Cooperative demo data (item prices and discount rules) -- Japanese scenario.
vdb.add_fact("リンゴ", "1個 150円")
vdb.add_fact("バナナ", "1房 200円")
vdb.add_fact("オレンジ", "1個 120円")
vdb.add_fact("雨の日キャンペーン", "雨の日は合計金額から10%オフ (0.9倍)")
vdb.add_fact("HAPPY50", "合計金額から50円引き")

# Knowledge entries -- we leave the price/discount entries untouched and use
# extra data to boost similarity-search accuracy. With these, a descriptive
# query like "a red round fruit" still resolves to the correct canonical item.
vdb.add_fact("リンゴの特徴", "リンゴは赤くて丸い甘い果物です")
vdb.add_fact("バナナの特徴", "バナナは黄色くて細長い湾曲した果物です")
vdb.add_fact("オレンジの特徴", "オレンジは橙色の球形の柑橘類の果物です")
vdb.add_fact("雨の表現", "土砂降り 雨天 雨の日 はすべて雨を表します。雨の日キャンペーンの対象です。")

model.add_synapse("VectorDB", vdb)

# Function that turns the input text into dummy IDs
def text_to_ids(text, max_len=5):
    # Simple demo-only conversion
    ids = [(ord(c) * 31) % vocab_size for c in text]
    if len(ids) < max_len:
        ids += [0] * (max_len - len(ids))
    return torch.tensor([ids[:max_len]]).to(device)

# Train the router for the demo
demo_data = [
    {"text": "15 * 4", "target": 1},
    {"text": "100 / 2", "target": 1},
    {"text": "2 + 2", "target": 1},
    {"text": "Japan", "target": 2},
    {"text": "CEO", "target": 2},
    {"text": "SynapticRouter", "target": 2},
    {"text": "Hello", "target": 0},
    # NOTE: Japanese greeting is intentionally kept so the router learns the demo input.
    {"text": "こんにちは", "target": 0},
    {"text": "How are you?", "target": 0},
]

optimizer = torch.optim.Adam(model.router.parameters(), lr=0.05)
criterion = nn.CrossEntropyLoss()

print("Training the router...")
for epoch in range(100):
    optimizer.zero_grad()
    loss = 0
    for d in demo_data:
        ids = text_to_ids(d["text"])
        x = model.embedding(ids)
        _, logits = model.router(x)
        loss += criterion(logits, torch.tensor([d["target"]]).to(device))
    loss.backward()
    optimizer.step()
print("Training complete!")

## Интерфейс чат-бота
Запустите ячейку ниже, чтобы открыть окно чата.
Попробуйте ввести арифметическое выражение (например, `15 * 4`), запрос знаний (например, `Japan`, `SynapticRouter`) или общее приветствие (например, `Hello`).

In [ ]:
# Build the chat UI
# NOTE: This UI is designed to drive a Japanese-language fruit-shop demo scenario
# (see Notebook header). User-facing button labels and placeholders stay in Japanese
# because the LLM prompts the router cooperates with are Japanese.
import html

response_area = widgets.HTML(
    value='<div><strong>SRA:</strong> Please type a message</div>',
    layout=widgets.Layout(
        width='100%',
        min_height='96px',
        border='1px solid #d0d7de',
        padding='12px',
        margin='0 0 8px 0',
    ),
)
chat_output = widgets.Output()
chat_history = []

text_input = widgets.Text(
    value='',
    placeholder='Type a message... (e.g.: 15 * 4, Japan, Hello)',
    description='You:',
    disabled=False,
    layout=widgets.Layout(width='80%'),
)

send_button = widgets.Button(
    description='Send',
    disabled=False,
    button_style='info', 
    tooltip='Click to send',
    icon='paper-plane'
)

# Quick-select example prompt buttons. Labels include the (Japanese) example
# prompts because the multi-stage cooperative routing demo expects Japanese input.
example_label = widgets.HTML("<b>Example prompts (click to send):</b>")
btn_layout = widgets.Layout(margin='2px', width='auto')

btn_ex1 = widgets.Button(description="Chat: こんにちは！自己紹介をして", layout=btn_layout, button_style='primary')
btn_ex2 = widgets.Button(description="Exact calc: 15 * 4 + 100", layout=btn_layout, button_style='success')
btn_ex3 = widgets.Button(description="Multi-stage cooperative: 赤い丸い果物3個と細長い黄色い果物2本、雨、HAPPY50", layout=btn_layout, button_style='warning')

def on_ex_clicked(b):
    if b == btn_ex1:
        # NOTE: Japanese demo prompt (the LLM is asked to reply naturally in the same language).
        text_input.value = "こんにちは！自己紹介をして。"
    elif b == btn_ex2:
        # NOTE: Japanese demo prompt that exercises Calculator+LLM cooperation.
        text_input.value = "次の計算をして。15 * 4 + 100"
    elif b == btn_ex3:
        # NOTE: Japanese demo prompt that exercises the multi-stage VDB+LLM+Calc pipeline.
        text_input.value = "赤い丸い果物を3個と、黄色くて細長い果物を2本買います。今日あいにく土砂降りなんだけど何かキャンペーンある？ あとシークレットクーポン『HAPPY50』も使いたいです。"
    on_send_clicked(None)

btn_ex1.on_click(on_ex_clicked)
btn_ex2.on_click(on_ex_clicked)
btn_ex3.on_click(on_ex_clicked)

examples_box = widgets.VBox([
    example_label,
    widgets.HBox([btn_ex1, btn_ex2]),
    widgets.HBox([btn_ex3])
])

def _format_response(text, routed_synapse=None):
    safe_text = html.escape(text).replace('\n', '<br>')
    route_label = f" <span style='color:#57606a'>(via {html.escape(routed_synapse)})</span>" if routed_synapse else ''
    return f"<div><strong>SRA:</strong>{route_label}<br>{safe_text}</div>"

def on_send_clicked(b):
    user_text = text_input.value.strip()
    if not user_text:
        return
        
    # Command handling
    if user_text.startswith('/'):
        cmd_parts = user_text.split(None, 1)
        cmd = cmd_parts[0].lower()
        args = cmd_parts[1].strip() if len(cmd_parts) > 1 else ""
        
        response_text = ""
        routed_synapse = "SystemCommand"
        
        if cmd == "/adddb":
            if "=" in args:
                k, v = args.split("=", 1)
                k = k.strip()
                v = v.strip()
                if k and v:
                    if "VectorDB" in model.synapses:
                        model.synapses["VectorDB"].add_fact(k, v)
                        response_text = f"Added fact to the vector DB: {k} -> {v}"
                    else:
                        response_text = "Error: VectorDB synapse not found."
                else:
                    response_text = "Error: key and value must not be empty."
            else:
                response_text = "Usage: /adddb key=value (e.g. /adddb US=Washington)"
                
        elif cmd == "/listdb":
            if "VectorDB" in model.synapses:
                db_content = model.synapses["VectorDB"].db
                if db_content:
                    lines = [f"- {k}: {v}" for k, v in db_content.items()]
                    response_text = "Vector DB contents:\n" + "\n".join(lines)
                else:
                    response_text = "The vector DB is empty."
            else:
                response_text = "Error: VectorDB synapse not found."
                
        elif cmd == "/deldb":
            k = args.strip()
            if k:
                if "VectorDB" in model.synapses:
                    vdb_module = model.synapses["VectorDB"]
                    if k in vdb_module.db:
                        del vdb_module.db[k]
                        response_text = f"Removed fact from the vector DB: {k}"
                    else:
                        response_text = f"Error: key '{k}' was not found in the vector DB."
                else:
                    response_text = "Error: VectorDB synapse not found."
            else:
                response_text = "Usage: /deldb key (e.g. /deldb US)"
                
        elif cmd == "/synapses":
            response_text = "Registered synapses:\n"
            for idx, name in enumerate(model.synapse_names):
                syn = model.synapses[name]
                is_neural = getattr(syn, "is_neural", True)
                syn_type = "Neural" if is_neural else "Rule-based/Retrieval"
                response_text += f"{idx + 1}. {name} ({syn_type})\n"
                
        elif cmd == "/clear":
            chat_history.clear()
            chat_output.clear_output()
            response_text = "Chat history cleared."
            
        elif cmd == "/help":
            response_text = (
                "Available commands:\n"
                "- /adddb key=value: add a new fact to the vector DB\n"
                "- /listdb: list every fact currently in the vector DB\n"
                "- /deldb key: remove the specified fact from the vector DB\n"
                "- /synapses: list every currently registered synapse\n"
                "- /clear: clear the chat history\n"
                "- /help: show this help message"
            )
        else:
            response_text = f"Unknown command: {cmd}\nType /help to see the available commands."
            
        with chat_output:
            print(f"You: {user_text}")
        response_area.value = _format_response(response_text, routed_synapse)
        text_input.value = ''
        return
        
    response_area.value = _format_response('Thinking...')
        
    with chat_output:
        print(f"You: {user_text}")
        
        # Turn the input into IDs and run routing + inference
        ids = text_to_ids(user_text)
        llm_messages = chat_history + [{"role": "user", "content": user_text}]
        outputs, routed_synapse = model.route_and_forward(
            ids,
            text_inputs=[user_text],
            llm_messages=[llm_messages],
        )

        # Display the response
        assistant_reply = outputs[0].split('] ', 1)[1]
        response_area.value = _format_response(assistant_reply, routed_synapse)
        if routed_synapse in ["GeneralLLM", "Cooperative(Calc+LLM)", "Cooperative(LLM+DB+Calc)"]:
            chat_history.append({"role": "user", "content": user_text})
            chat_history.append({"role": "assistant", "content": assistant_reply})
            del chat_history[:-8]
        
    text_input.value = ''

send_button.on_click(on_send_clicked)
text_input.continuous_update = False
def _on_value_change(change):
    if change.get('name') == 'value' and change.get('type') == 'change':
        on_send_clicked(None)
text_input.observe(_on_value_change, names='value')

display(widgets.VBox([response_area, examples_box, widgets.HBox([text_input, send_button]), chat_output]))
